<a href="https://colab.research.google.com/github/IshiJess/BITACORA-DE-SECUENCIACION--Jessica-Noemi-Salcedo-Alferez-CaBioinf2027/blob/main/Bitacora_Secuenciacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <span style="color:#A230A4;">**Bitácora de Análisis Bioinformático de secuencias**</span>

**Estudiante:** Jessica Noemi Salcedo Alferez  
**Fecha:** Septiembre 2026  
**Entorno de ejecución:** Ubuntu Linux sobre windows


### ***1.Recibimos los archivos y verificamos la integridad de estos.***
El análisis comienza con la verificación de la integridad de los archivos FastQ "crudos" mediante el cálculo de huellas digitales LLAMADAS MD5 (hashes) para asegurar que no ocurriera corrupción durante la transferencia de datos.

* **Ruta de origen:** `/projects/share/CURSO/`
* **Hacemos un soft Link para la ruta de origen:** `ln -s /projects/share/CURSO/ CURSO`
* **Archivos:** `Ch_strain_1.fastq.gz` y `Ch_strain_2.fastq.gz`


### ***2. Crear un directorio de trabajo y moverse a él***
mkdir -p /projects/share/CURSO/analisis_calidad

cd /home/jsalcedo/analisis_calidad


### ***3. Generar los hashes MD5 de los archivos recibidos***
md5sum /projects/share/CURSO/Ch_strain_1.fastq.gz > hashes_iniciales.md5
md5sum /projects/share/CURSO/Ch_strain_2.fastq.gz >> hashes_iniciales.md5

## 2. Descompresión de Archivos gz
Para facilitar la visualización directa de nuestras secuencias en un formato de archivo plano (aunque la mayoría de las herramientas bioinformáticas modernas aceptan `.gz`).

### Comandos ejecutados:
```bash
# Descomprimir
gunzip -c /projects/share/CURSO/Ch_strain_1.fastq.gz > Ch_strain_1.fastq
gunzip -c /projects/share/CURSO/Ch_strain_2.fastq.gz > Ch_strain_2.fastq

# Verificar que los archivos descomprimidos se crearon correctamente
ls -lh Ch_strain_*.fastq
```

## 3. Visualización de los Archivos en formato FastQ
Inspección visual de la estructura interna del formato FastQ (identificador, secuencia, línea de separación, y valores de calidad Phred).

Para este paso, aprendí como hacerlo gracias a la asesoría de mi compañero egresado, Saúl Villa. Quien nos enseñó a hacer esto.
### Comandos ejecutados:
```bash
# Visualizar las primeras 20 líneas de ntro fastq (equivalente a las primeras 5 lecturas) del archivo R1
head -n 20 Ch_strain_1.fastq

# Visualizar las primeras 20 líneas del archivo R2
head -n 20 Ch_strain_2.fastq

# Contar el número total de líneas para estimar el número de lecturas (líneas totales divididas por 4)
wc -l Ch_strain_1.fastq Ch_strain_2.fastq
```

## 4. Control de Calidad Inicial (QC)
Ejecución de un diagnóstico automatizado utilizando FastQC para evaluar métricas clave como la calidad por base, contenido de GC, niveles de duplicación y presencia de adaptadores.

### Comandos ejecutados:
```bash
# Crear un directorio exclusivo para los reportes de calidad
mkdir -p reportes_fastqc

# Ejecutar FastQC en los archivos originales comprimidos (ahorra espacio y tiempo)
fastqc -o reportes_fastqc/ /projects/share/CURSO/Ch_strain_1.fastq.gz /projects/share/CURSO/Ch_strain_2.fastq.gz

```
*Nota: Los archivos `.html` resultantes se descargan localmente para su revisión visual en cualquier navegador web.*


## 5. Preprocesamiento (Filtrado y Recorte de Lecturas)
Eliminación de bases de baja calidad en los extremos (Trimming), remoción de secuencias de adaptadores contaminantes y descarte de lecturas excesivamente cortas. En este comando se utiliza `fastp`

### Opción A: Preprocesamiento con `fastp`
```bash
# Crear directorio para los datos limpios, usamos -p por si el directorio ya existe, no me aparece error
mkdir -p datos_Fastp

# Ejecutar fastp para lecturas pareadas (Paired-End)
fastp   -i /CURSO/Ch_strain_1.fastq.gz   -I /CURSO/Ch_strain_2.fastq.gz   -o datos_Fastp/Ch_strain_1.clean.fastq.gz   -O datos_preprocesados/Ch_strain_2.clean.fastq.gz   -h datos_preprocesados/reporte_fastp.html   -j datos_preprocesados/reporte_fastp.json  
```

## 6. Control de Calidad Post-Preprocesamiento
Hacemos una validación final para asegurar que el proceso de filtrado eliminó con éxito las lecturas problemáticas y los adaptadores.

### Comandos ejecutados:
```bash
# Crear directorio para el segundo control de calidad
mkdir -p reportes_fastqc_post

# Ejecutar FastQC sobre las lecturas limpias pareadas
fastqc -o reportes_fastqc_post/ datos_preprocesados/Ch_strain_1.clean.fastq.gz datos_preprocesados/Ch_strain_2.clean.fastq.gz
```



## 7. Mapear los genomas con ayuda de Bowtie 2 y bwa
Si tenemos una referencia podemos mapear nuestros reads a ella. En este caso tenemos la referencia Ch_200k_ref.fna. Uno de los programas más populares para mapear a referencia es Bowtie2

  _*Nota: En el caso de nuestro cluster Bowtie2 está instalado bajo un ambiente de conda. Para activarlo 'conda activate bowtie2' Si marca error 'conda init' para poder usar tu cuenta con conda. Al finalizar bowtie2 habrá que desactivar el ambiente 'conda deactivate'.*_

Realiza un mapeo de una biblioteca pareada que proviene de la secuenciación de una bacteria.


### *1. Crear y moverse al directorio de trabajo*
```bash
mkdir -p MapBowtie2
cd MapBowtie2
```
#### Nota. Revisa los archivos
** 3- ¿Cuánto mide nuestra referencia?**
```bash
$ tail -1 Ch_200kref.fna | wc -c
200001
```
*4- ¿Cuántas secuencias tenemos en los archivos fastq?*
```bash
$ awk 'NR%4==2{print}' Ch_strain_1.fastq | wc -l
```
### 2. Construcción del índice de referencia*
```bash
bowtie2-build -f ../CURSO/Ch_200kref.fna Ch_200
```
### *3. Mapeo de lecturas pareadas*
```bash
bowtie2 --maxins 1000 -x Ch_200 -1 ../CURSO/Ch_strain_1.fastq -2 ../CURSO/Ch_strain_2.fastq -S ChVsCh.sam
```
### *4. Filtrado del archivo SAM (conservar solo reads alineados)*
```bash
awk '$3!="*"' ChVsCh.sam > filtChVsCh.sam
```
### 5. *Transferencia de resultados a nuestra computadora fuera de Gaia (scp)*
Para respaldar y analizar los resultados en nuestra computadora local, se descan los archivos mediante el protocolo scp a través del puerto configurado 10022 desde Windows PowerShell:   

```bash
PowerShellscp -P 10022 jsalcedo@132.247.172.26:/home/jsalcedo/Proyect1/MapBowtie2/filtChVsCh.sam C:\Users\Armando\Downloads\
scp -P 10022 jsalcedo@132.247.172.26:/home/jsalcedo/Proyect1/MapBowtie2/ChVsCh.sam C:\Users\Armando\Downloads\
scp -P 10022 jsalcedo@132.247.172.26:/home/jsalcedo/Proyect1/CURSO/Ch_strain_1.fastq C:\Users\Armando\Downloads\_fastp.html   -j datos_preprocesados/reporte_fastp.json  
```

## Comandos para BWA
### Crear índice a partir del archivo de referencia
```bash
bwa index /projects/share/CURSO/Ch_200kref.fna -p bwaIndex
```

#### mapear fastq reads a la referencia
```bash
bwa mem /projects/share/CURSO/Ch_200kref.fna /projects/share/CURSO/Ch_strain_1.fastq /projects/share/CURSO/Ch_strain_2.fastq > BWAChVsCh.sam
```
#### calcular estadísticas
```bash
samtools flagstat ChVsCh.sam
```

#### filtrar dejando sólo las líneas con alineamientos
```bash
awk '$3!="*"' BWAChVsCh.sam > filtBWAChVsCh.sam
```

#### bajar archivo de referencia y de mapeo y visualizarlo con tablet
```bash
PS C:\Users\Armando> scp -P 10022 jsalcedo@132.247.172.26:/home/jsalcedo/Proyect1/MapBWA/filtBWAChVsCh.sam C:\Users\Armando\Downloads\
```

## 8. Procesamiento Avanzado de Alineamientos (SAMtools y Picard)
Conversión de formatos y métricas de inserción a partir del archivo SAM filtrado:

### Comandos ejecutados:

####  Convertir SAM a BAM
```bash
samtools view -b -o filtChVsCh.bam filtChVsCh.sam
```

#### Generar estadísticas con samtools

```bash
samtools stats filtChVsCh.bam > insert_size_stats.txt
```

#### Recopilar métricas de tamaño de inserción con Picard

```bash
picard CollectInsertSizeMetrics I=filtChVsCh.bam

O=insert_size_metrics.txt H=insert_size_histogram.pdf M=0.5
```

---
**Fin de mi Bitácora para nuestra secuenciación y análisis bioinformático.** Los archivos limpios están ubicados en gaia en mi home `home/jsalcedo/Proyect1` y ya están listos para el ensamble o alineamiento genómico o cualquier análisis bioinformático de nuestro interés :D